In [23]:
from ..middleWare import *
load_dotenv(override=True)

DEEPSEEK_API_KEY=os.getenv('DEEPSEEK_API_KEY')
DEEPSEEK_BASE_URL=os.getenv('DEEPSEEK_BASE_URL')
model=init_chat_model(
    model='deepseek-v4-flash',
    model_provider='deepseek',
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
    extra_body={'thinking':{"type":'disabled'}},
)

In [25]:
#自定义中间件与hook钩子函数的应用
#hook函数有六个分两种，节点类方法和包装类方法
#节点类包括before_model,before_agent,after_model,after_agent，这类方法一般不干预流程，用于更新状态，可以不用返回值
#包装类包括wrap_tool_call和wrap_model_call，这类方法接管流程调用前后，一般干预流程，需要返回响应值
#自定义中间件通过组合hook函数实现，有两种实现方法，一种基于装饰器，一种基于继承类
#1.基于装饰器实现
#基于装饰器实现，一般用于单独定义方法，名字可以自定义，也可以放在类中使用，但不能是实例方法self，传入参数时需要传入函数
#节点类参数包括Agentstate和Runtime，其中Agentstate包含了整个调用过程的状态信息，包括messages列表，可以用于更新状态
#包装类参数包括request,handler参数，返回response，可以拦截请求进行修改后，用handler处理器处理请求后返回响应，针对模型和工具类似
class MyMiddleware:

    @before_model
    def before_model_log(state:AgentState, runtime: Runtime) -> dict[str, Any]|None:
        logger.debug("before_model_log")
        state["messages"][-1].content += " --before_model_log--"
        return None

    @after_model
    def after_model_log( state:AgentState, runtime: Runtime) -> dict[str, Any]|None:
        logger.debug("after_model_log")
        state["messages"][-1].content += " --after_model_log--"
        return None

# 传入实例
myagent = create_agent(
    model=model,
    middleware=[MyMiddleware().before_model_log,MyMiddleware.after_model_log],
)
response=myagent.invoke({
    'messages':HumanMessage('你好')
})
for msg in response['messages']:
    rprint(msg)


2026-07-24 15:54:12.245 | DEBUG    | __main__:before_model_log:14 - before_model_log
2026-07-24 15:54:13.117 | DEBUG    | __main__:after_model_log:20 - after_model_log


HumanMessage(
    content='你好 --before_model_log--',
    additional_kwargs={},
    response_metadata={},
    id='d1dae0af-320c-4117-9d7e-21cd09a437a4'
)

AIMessage(
    content='你好！有什么可以帮你的吗？😊 --after_model_log--',
    additional_kwargs={'refusal': None},
    response_metadata={
        'token_usage': {
            'completion_tokens': 10,
            'prompt_tokens': 10,
            'total_tokens': 20,
            'completion_tokens_details': None,
            'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0},
            'prompt_cache_hit_tokens': 0,
            'prompt_cache_miss_tokens': 10
        },
        'model_provider': 'deepseek',
        'model_name': 'deepseek-v4-flash',
        'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402',
        'id': 'b7909790-49f1-4a84-ab3c-28ff242c72ab',
        'finish_reason': 'stop',
        'logprobs': None
    },
    id='lc_run--019f931e-1d99-73a3-aa90-9455093155c0-0',
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 10,
        'output_tokens': 10,
        'total_tokens': 20,
        'input_token_details': {'cache_read': 0},
        'output_token_details': {}
    }
)

In [34]:
#2.基于类实现
#一般用于多个钩子方法组合使用，集成AgentMiddleware类，方法名固定
class MyMiddleware2(AgentMiddleware):
    def __init__(self):
        super().__init__()
    @hook_config(can_jump_to=['end'])
    def before_model(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        logger.info("before_model")
        text = state["messages"][-1].content
        if 'end' in text:
            return {
                'jump_to': 'end',
            }
        state["messages"][-1].content += " --before_model--"
        return None
    def wrap_model_call(
        self,
        request: ModelRequest,
        handler: Callable[[ModelRequest], ModelResponse],
    ) -> ModelResponse|None :
        logger.info("wrap_model_call")
        request.messages[-1].content += " --wrap_model_call--"
        response=handler(request)
        response.result[0].content += " --wrap_model_call--"
        return response
myagent2=create_agent(
    model=model,
    middleware=[MyMiddleware2()],
)
response=myagent2.invoke({
    'messages':HumanMessage('你好')
})
for msg in response['messages']:
    rprint(msg)

2026-07-24 16:41:32.684 | INFO     | __main__:before_model:11 - before_model
2026-07-24 16:41:32.684 | INFO     | __main__:wrap_model_call:24 - wrap_model_call


HumanMessage(
    content='你好 --before_model-- --wrap_model_call--',
    additional_kwargs={},
    response_metadata={},
    id='46917afb-b90c-4a99-98d9-4cad929f9dbb'
)

AIMessage(
    content='你好！😊\n\n看起来你在命令中使用了 `--before_model` 和 `--wrap_model_call` 
这两个参数，但我不太确定你想做什么。你能告诉我更多关于你的需求吗？\n\n是想要：\n1. 了解一下这些参数的作用？\n2. 
或者有其他特定任务需要我帮忙完成？\n\n请详细描述一下，我会尽力帮你解答！🚀 --wrap_model_call--',
    additional_kwargs={'refusal': None},
    response_metadata={
        'token_usage': {
            'completion_tokens': 77,
            'prompt_tokens': 14,
            'total_tokens': 91,
            'completion_tokens_details': None,
            'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0},
            'prompt_cache_hit_tokens': 0,
            'prompt_cache_miss_tokens': 14
        },
        'model_provider': 'deepseek',
        'model_name': 'deepseek-v4-flash',
        'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402',
        'id': 'a18dd2df-6a74-4655-b586-d0f2c6dde843',
        'finish_reason': 'stop',
        'logprobs': None
    },
    id='lc_run--019f9349-7511-7e93-ba44-439699888c46-0',
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 14,
        'output_tokens': 77,
        'total_tokens': 91,
        'input_token_details': {'cache_read': 0},
        'output_token_details': {}
    }
)